In [ ]:
import os
import re
import glob
import pandas as pd
import jiwer
from transformers import pipeline

GROUND_TRUTH = [
    (
        "Please call Stella. Ask her to bring these things with her from the store: "
        "Six spoons of fresh snow peas, five thick slabs of blue cheese, and maybe a "
        "snack for her brother Bob. We also need a small plastic snake and a big toy "
        "frog for the kids. She can scoop these things into three red bags, and we "
        "will go meet her Wednesday at the train station."
    )
    for _ in range(60)
]

US_DIR = "us_clips"
INDIA_DIR = "india_clips"
MODEL_NAME = "Meta MMS-1B-all"
MODEL_ID = "facebook/mms-1b-all"

In [ ]:
def normalize(text: str) -> str:
    text = text.lower()
    text = re.sub(r"[^\w\s]", "", text)
    digit_words = {
        "0": "zero", "1": "one", "2": "two", "3": "three", "4": "four",
        "5": "five", "6": "six", "7": "seven", "8": "eight", "9": "nine",
    }
    text = re.sub(r"\b\d\b", lambda m: digit_words.get(m.group(), m.group()), text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


print(f"Loading {MODEL_NAME}... (first run downloads ~3.5GB, be patient)")
pipe = pipeline("automatic-speech-recognition", model=MODEL_ID)
# MMS defaults to its English adapter automatically for English audio.
# To be explicit (and if you extend this project to other languages later):
# pipe.model.load_adapter("eng")


def transcribe_folder(folder_path: str, group_name: str):
    results = []
    files = sorted(glob.glob(os.path.join(folder_path, "*.mp3")))
    if not files:
        print(f"  WARNING: no mp3 files found in {folder_path}")

    for i, filepath in enumerate(files):
        output = pipe(filepath)
        prediction = output["text"].strip()
        results.append({
            "group": group_name,
            "file": os.path.basename(filepath),
            "ground_truth": GROUND_TRUTH,
            "prediction_raw": prediction,
            "prediction_norm": normalize(prediction),
        })
        print(f"[{MODEL_NAME} | {group_name} {i + 1}/{len(files)}] {os.path.basename(filepath)}")
        print(f"  {prediction}\n")

    return results

In [ ]:
print("\n--- Transcribing US clips ---\n")
us_results = transcribe_folder(US_DIR, "US")

In [ ]:
print("\n--- Transcribing Indian clips ---\n")
india_results = transcribe_folder(INDIA_DIR, "India")

In [ ]:
all_results = us_results + india_results
df = pd.DataFrame(all_results)
df.to_csv("meta_results.csv", index=False)
print("Saved meta_results.csv")

gt_norm = [normalize(text) for text in GROUND_TRUTH]

us_wer = jiwer.wer(
    gt_norm,
    [normalize(r["prediction_raw"]) for r in us_results]
)

india_wer = jiwer.wer(
    gt_norm,
    [normalize(r["prediction_raw"]) for r in india_results]
)

In [ ]:
print(f"\n========== FINAL RESULTS: {MODEL_NAME} ==========")
print(f"US English WER:     {us_wer:.4f} ({us_wer*100:.2f}%)" if us_wer is not None else "US: no clips found")
print(f"Indian English WER: {india_wer:.4f} ({india_wer*100:.2f}%)" if india_wer is not None else "India: no clips found")
print("===================================================")
